In [4]:
import json
import bm25s

#load data
with open(r'C:\Users\anandha.kumar\OneDrive - ClaySys Technologies\Desktop\Code Base\Global search\data\processed\restructured_dataset.json', 'r') as f:
    dataset_json = json.load(f)

# Flatten
corpus = []
intent_mapping = []
for intent, phrases in dataset_json.items():
    for phrase in phrases:
        corpus.append(phrase)
        intent_mapping.append(intent)

#tokenize the data
def tokenize(text):
    return text.lower().split()

corpus_tokens = [tokenize(doc) for doc in corpus]

# Retriver 
retriever = bm25s.BM25()
retriever.index(corpus_tokens)

# predit function

def predict_top_n_intents(query, top_n=1):
    query_tokens = tokenize(query)
    
    results = retriever.retrieve([query_tokens], k=top_n)
    
    predicted_intents = []
    actual_k = min(top_n, results.scores.shape[1])
    
    for i in range(actual_k):
        score = results.scores[0][i]
        
        if score > 0:
            matched_doc_index = results.documents[0][i]
            intent = intent_mapping[matched_doc_index]
            
            if intent not in predicted_intents:
                predicted_intents.append(intent)
                
    return predicted_intents

# Testing
test_queries = [
    "I want to pay my electricity bill",
    "where is my check deposit history?",
    "how to update my email address"
]

print("--- Intent Prediction Results ---")
for q in test_queries:
    top_intents = predict_top_n_intents(q, top_n=1)
    print(f"Query: '{q}' -> Top Predicted Intents: {top_intents}")

--- Intent Prediction Results ---


Query: 'I want to pay my electricity bill' -> Top Predicted Intents: ['BillPay']


Query: 'where is my check deposit history?' -> Top Predicted Intents: ['CheckDeposit']


Query: 'how to update my email address' -> Top Predicted Intents: ['MyProfile']


In [ ]:
import pandas as pd

df = pd.read_json(r"C:\Users\anandha.kumar\OneDrive - ClaySys Technologies\Desktop\Code Base\Global search\data\raw\evaluation_dataset.json")
df_prediction = df.copy()

excluded_intents = ['OrderaCheck', 'ScheduledInternalTransfer', "SpecialCharacters", "ApplyMortgage", "Navigate", "Transfers"]
df_prediction = df_prediction[~df_prediction['actual'].isin(excluded_intents)].copy()

TOP_N = 5

def evaluate_row(row):
    query = row['query']  
    actual_intent = row['actual']
    

    top_n_intents = predict_top_n_intents(query, top_n=TOP_N)

    while len(top_n_intents) < TOP_N:
        top_n_intents.append("None")
        
    ranked_predictions = [{"rank": idx + 1, "intent": intent} for idx, intent in enumerate(top_n_intents)]
  
    is_correct = actual_intent in top_n_intents and actual_intent != "None"
    
    return pd.Series([ranked_predictions, is_correct])

# Create Columns
df_prediction[['predictions', 'correct_top_5']] = df_prediction.apply(evaluate_row, axis=1)

# Calculate Accuracy
correct_top5_predictions = df_prediction['correct_top_5'].sum()
top5_accuracy = correct_top5_predictions / len(df_prediction)

print(f"Total Queries: {len(df_prediction)}")
print(f"Top-{TOP_N} Correct Predictions: {correct_top5_predictions}")
print(f"Top-{TOP_N} Accuracy: {top5_accuracy:.2%}")

Total Queries: 436
Top-5 Correct Predictions: 433
Top-5 Accuracy: 99.31%


In [16]:
df_prediction.to_excel(r"C:\Users\anandha.kumar\OneDrive - ClaySys Technologies\Desktop\Code Base\Global search\data\processed\bm25s_predictions.xlsx")